# QGNN

In [ ]:
#download
%pip install -q \
    currencyconverter \
    forex-python \
    kagglehub \
    optuna \
    pennylane \
    torch-geometric \
    datasets \
    networkx \
    polars \
    dataframes \
    fastapi

Import IBM dataset

In [5]:
import os
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import pandas as pd

# Prefer local subsampled CSV if present; otherwise download + preprocess
csv_file = Path("ibmdata.csv")
if not csv_file.is_file():
    path = kagglehub.dataset_download(
        "ealtman2019/ibm-transactions-for-anti-money-laundering-aml"
    )
    print("Path to dataset files:", path)

    DATA_DIR = path
    print("Using DATA_DIR:", DATA_DIR)
    print("Files:", os.listdir(DATA_DIR))

    SMALL_TRANS_FILE = os.path.join(DATA_DIR, "HI-Small_Trans.csv")
    SMALL_ACCOUNTS_FILE = os.path.join(DATA_DIR,"HI-Small_accounts.csv")
    SMALL_PATTERNS_FILE = os.path.join(DATA_DIR, "HI-Small_Patterns.txt")

    MEDIUM_TRANS_FILE = os.path.join(DATA_DIR, "HI-Medium_Trans.csv")
    MEDIUM_ACCOUNTS_FILE = os.path.join(DATA_DIR, "HI-Medium_accounts.csv")
    MEDIUM_PATTERNS_FILE = os.path.join(DATA_DIR, "HI-Medium_Patterns.txt")

    LARGE_TRANS_FILE = os.path.join(DATA_DIR, "HI-Large_Trans.csv")
    LARGE_ACCOUNTS_FILE = os.path.join(DATA_DIR, "HI-Large_accounts.csv")
    LARGE_PATTERNS_FILE = os.path.join(DATA_DIR, "HI-Large_Patterns.txt")

    # Raw LI-Small transactions — run Data Analysis on this before preprocessing
    df = pd.read_csv(MEDIUM_TRANS_FILE)
    display(df.head())
    df.info()
    


    # Strategic Subsampling
    # Start with all fraud, grow by adding transactions connected to the current
    # set (same User or Merchant Name), and only randomly fill if still under the
    # 1% fraud-rate budget after connected candidates are exhausted.
    print("\n--- Starting Strategic Subsampling ---")
    #add unique id to each account+bank combo
    df['From Bank'] = df['From Bank'].astype(str)
    df['To Bank'] = df['To Bank'].astype(str)
    df['sender_id'] = df['Account'] + '_' + df['From Bank']
    df.head()


    #add unique id to each account+bank combo
    df['receiver_id'] = df['Account.1'] + '_' + df['To Bank']
    df.head()

    MIN_FRAUD_RATE = 0.05  # at least 1% fraudulent
    RNG = 42

    fraud_rows = df[df["Is Laundering"] == 1]
    n_fraud = len(fraud_rows)
    max_total = int(n_fraud / MIN_FRAUD_RATE)
    max_non_fraud = max(0, max_total - n_fraud)

    selected_idx = set(fraud_rows.index)
    sender_ids = set(fraud_rows["sender_id"])
    receiver_ids = set(fraud_rows["receiver_id"])
    remaining_slots = max_non_fraud

    # Grow via connectivity while we still have room under the fraud-rate cap
    while remaining_slots > 0:
        not_selected = ~df.index.isin(selected_idx)
        connected = df.loc[
            not_selected
            & (
                df["sender_id"].isin(sender_ids)
                | df["receiver_id"].isin(receiver_ids)
            )
        ]

        if connected.empty:
            break

        if len(connected) <= remaining_slots:
            selected_idx.update(connected.index)
            sender_ids.update(connected["sender_id"].tolist())
            receiver_ids.update(connected["receiver_id"].tolist())
            remaining_slots -= len(connected)
        else:
            # More connected rows than we can take — sample from connected only
            take = connected.sample(n=remaining_slots, random_state=RNG)
            selected_idx.update(take.index)
            remaining_slots = 0

    # If still under budget and nothing else is connected, fill randomly
    if remaining_slots > 0:
        leftover = df.loc[~df.index.isin(selected_idx)]
        if len(leftover) > remaining_slots:
            leftover = leftover.sample(n=remaining_slots, random_state=RNG)
        selected_idx.update(leftover.index)

    data = df.loc[sorted(selected_idx)].reset_index(drop=True)

    fraud_rate = data["Is Laundering"].mean() * 100
    print(
        f"rows={len(data):,} | "
        f"counts={data['Is Laundering'].value_counts().to_dict()} | "
        f"fraud_rate={fraud_rate:.2f}%"
    )
    df = data
    data.to_csv(csv_file, index=False)
else:
    print(f"Found existing {csv_file.resolve()} — loading instead of downloading")
    df = pd.read_csv(csv_file)





Found existing /Users/lachlanbruce/Documents/QDetect_CODE/Q-Detect/MachineLearning/Model_A_Lachlan/ibmdata.csv — loading instead of downloading


# Data Analysis

In [4]:
df["From Bank"].value_counts()
df["To Bank"].value_counts()

To Bank
0          2587
11         2571
12         2426
20         2358
70         1929
           ... 
2125122       1
1215972       1
1202316       1
217801        1
209145        1
Name: count, Length: 6478, dtype: int64

In [3]:
print(df["Payment Format"].value_counts())

Payment Format
Cheque          303088
Credit Card     195156
Cash            115965
ACH              63124
Bitcoin          11666
Reinvestment      8025
Wire              7576
Name: count, dtype: int64


In [4]:
# Check whether the same Account ID can appear under multiple banks

# Pair account + bank from both sides of each transaction
send = df[["Account", "From Bank"]].rename(
    columns={"Account": "account", "From Bank": "bank"}
)
recv = df[["Account.1", "To Bank"]].rename(
    columns={"Account.1": "account", "To Bank": "bank"}
)

pairs = pd.concat([send, recv], ignore_index=True).drop_duplicates()

banks_per_account = pairs.groupby("account")["bank"].nunique()
multi_bank = banks_per_account[banks_per_account > 1]

print(f"Unique accounts: {len(banks_per_account):,}")
print(f"Accounts under multiple banks: {len(multi_bank):,}")

if len(multi_bank) > 0:
    print(f"Max banks for one account: {multi_bank.max()}")
    print("\nTop examples:")
    display(multi_bank.sort_values(ascending=False).head(10))

    # Inspect one example
    example = multi_bank.index[0]
    print(f"\nBanks for account {example}:")
    display(pairs.loc[pairs["account"] == example].sort_values("bank"))
else:
    print("Each account ID maps to exactly one bank in this dataset.")

Unique accounts: 204,137
Accounts under multiple banks: 0
Each account ID maps to exactly one bank in this dataset.


In [5]:
# --- Data structure checks on RAW IBM AML LI-Small (before preprocessing) ---



EXPECTED_COLUMNS = [
    "Timestamp", "From Bank", "Account", "To Bank", "Account.1",
    "Amount Received", "Receiving Currency", "Amount Paid",
    "Payment Currency", "Payment Format", "Is Laundering", "sender_id", "receiver_id"
]
print("Shape (rows, columns):", df.shape)
print("Approximate memory (MB, shallow):", round(df.memory_usage(deep=False).sum() / 1e6, 2))
print()

missing_cols = [c for c in EXPECTED_COLUMNS if c not in df.columns]
extra_cols = [c for c in df.columns if c not in EXPECTED_COLUMNS]
print("Missing expected columns:", missing_cols if missing_cols else "none")
print("Unexpected columns:", extra_cols if extra_cols else "none")

print("Fraud Rate")
print(df["Is Laundering"].value_counts(normalize=True) * 100)

print("Column dtypes:")
print(df.dtypes)
print()

print("First 5 rows:")
display(df.head())
print()

# Self-transfers (same account+bank both sides) — motivate graph entity IDs
self_xfer = (
    (df["Account"] == df["Account.1"]) & (df["From Bank"] == df["To Bank"])
)
print(
    f"Self-transfers (same Account and Bank both sides): "
    f"{self_xfer.sum():,} ({100 * self_xfer.mean():.2f}%)"
)
print()

print("Missing values per column:")
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(3)
missing_summary = pd.DataFrame({"missing_count": missing, "missing_%": missing_pct})
display(missing_summary[missing_summary["missing_count"] > 0])
if missing_summary["missing_count"].sum() == 0:
    print("No missing values found.")
print()

print("Unique values (cardinality) for key columns:")
for col in [
    "From Bank", "To Bank", "Account", "Account.1",
    "Receiving Currency", "Payment Currency", "Payment Format", "Is Laundering",
]:
    print(f"  {col:20s}: {df[col].nunique(dropna=False):,}")
print()

print("Is Laundering label distribution:")
label_counts = df["Is Laundering"].value_counts(dropna=False)
label_pct = df["Is Laundering"].value_counts(normalize=True, dropna=False).mul(100).round(4)
print(label_counts)
print(label_pct.astype(str) + "%")
print()

print("Payment Format distribution:")
print(df["Payment Format"].value_counts(dropna=False))
print()

print("Receiving Currency / Payment Currency:")
print("Receiving Currency:", df["Receiving Currency"].value_counts().to_dict())
print("Payment Currency:  ", df["Payment Currency"].value_counts().to_dict())
same_currency = (df["Receiving Currency"] == df["Payment Currency"]).mean()
print(f"Same receiving & payment currency: {100 * same_currency:.2f}%")
print()

from_banks = set(df["From Bank"].unique())
to_banks = set(df["To Bank"].unique())
bank_overlap = from_banks & to_banks
print("From Bank / To Bank overlap:")
print(f"  Unique From Bank: {len(from_banks):,}")
print(f"  Unique To Bank:   {len(to_banks):,}")
print(
    f"  Overlapping IDs:  {len(bank_overlap):,} "
    f"({100 * len(bank_overlap) / len(from_banks):.2f}% of From, "
    f"{100 * len(bank_overlap) / len(to_banks):.2f}% of To)"
)
print()

from_accts = set(df["Account"].unique())
to_accts = set(df["Account.1"].unique())
acct_overlap = from_accts & to_accts
print("Account / Account.1 overlap:")
print(f"  Unique Account:   {len(from_accts):,}")
print(f"  Unique Account.1: {len(to_accts):,}")
print(
    f"  Overlapping IDs:  {len(acct_overlap):,} "
    f"({100 * len(acct_overlap) / len(from_accts):.2f}% of Account, "
    f"{100 * len(acct_overlap) / len(to_accts):.2f}% of Account.1)"
)
print()

# Timestamps: parse and report span (needed for temporal AML patterns)
ts = pd.to_datetime(df["Timestamp"], format="%Y/%m/%d %H:%M", errors="coerce")
print(f"Timestamp parse failures: {ts.isna().sum():,}")
print(f"Timestamp range: {ts.min()} → {ts.max()}")
print(f"Unique calendar days: {ts.dt.normalize().nunique():,}")
print()

# Amounts: same-currency subset only (mixed-currency describe is not comparable)

same_ccy = df["Receiving Currency"] == df["Payment Currency"]
print("Amount Received describe (same-currency rows):")
print(df.loc[same_ccy, "Amount Received"].describe())
print()
print("Amount Paid describe (same-currency rows):")
print(df.loc[same_ccy, "Amount Paid"].describe())
print()
print(
    f"Amount Received == Amount Paid (all rows): "
    f"{100 * (df['Amount Received'] == df['Amount Paid']).mean():.2f}%"
)

mismatch = df[df["Amount Received"] != df["Amount Paid"]]
print(f"Rows where Amount Paid != Amount Received: {len(mismatch):,}")
print("Sample of amount mismatches (5 rows):")
display(
    mismatch[
        [
            "Amount Received", "Receiving Currency",
            "Amount Paid", "Payment Currency", "Is Laundering",
        ]
    ].head(5)
)
print()

n_dup_full = int(df.duplicated().sum())
print(f"Duplicate rows (full table): {n_dup_full:,}")
print()

print("Currencies in the dataset:")
print(sorted(df["Receiving Currency"].dropna().unique().tolist()))
print(sorted(df["Payment Currency"].dropna().unique().tolist()))





Shape (rows, columns): (704600, 13)
Approximate memory (MB, shallow): 132.35

Missing expected columns: none
Unexpected columns: none
Fraud Rate
Is Laundering
0    95.0
1     5.0
Name: proportion, dtype: float64
Column dtypes:
Timestamp                 str
From Bank               int64
Account                   str
To Bank                 int64
Account.1                 str
Amount Received       float64
Receiving Currency        str
Amount Paid           float64
Payment Currency          str
Payment Format            str
Is Laundering           int64
sender_id                 str
receiver_id               str
dtype: object

First 5 rows:


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering,sender_id,receiver_id
0,2022/09/01 00:26,0,800CCE690,235792,81B2A80E0,27.56,US Dollar,27.56,US Dollar,Credit Card,0,800CCE690_0,81B2A80E0_235792
1,2022/09/01 00:03,70,100428660,20,800070AC0,5105.92,US Dollar,5105.92,US Dollar,Cheque,0,100428660_70,800070AC0_20
2,2022/09/01 00:22,11,8000711E0,11,8000711E0,1135598.87,US Dollar,1135598.87,US Dollar,Reinvestment,0,8000711E0_11,8000711E0_11
3,2022/09/01 00:27,0,800104F90,0,800104F90,614545.68,US Dollar,614545.68,US Dollar,Reinvestment,0,800104F90_0,800104F90_0
4,2022/09/01 00:16,20,800106EE0,20,800106EE0,105352.58,US Dollar,105352.58,US Dollar,Reinvestment,0,800106EE0_20,800106EE0_20



Self-transfers (same Account and Bank both sides): 14,626 (2.08%)

Missing values per column:


,missing_count,missing_%


No missing values found.

Unique values (cardinality) for key columns:
  From Bank           : 8,669
  To Bank             : 6,478
  Account             : 53,360
  Account.1           : 170,071
  Receiving Currency  : 15
  Payment Currency    : 15
  Payment Format      : 7
  Is Laundering       : 2

Is Laundering label distribution:
Is Laundering
0    669370
1     35230
Name: count, dtype: int64
Is Laundering
0    95.0%
1     5.0%
Name: proportion, dtype: str

Payment Format distribution:
Payment Format
Cheque          303088
Credit Card     195156
Cash            115965
ACH              63124
Bitcoin          11666
Reinvestment      8025
Wire              7576
Name: count, dtype: int64

Receiving Currency / Payment Currency:
Receiving Currency: {'US Dollar': 268941, 'Euro': 177445, 'Yuan': 51777, 'Shekel': 26013, 'UK Pound': 21535, 'Ruble': 21458, 'Canadian Dollar': 20032, 'Yen': 19468, 'Australian Dollar': 17866, 'Rupee': 17011, 'Swiss Franc': 15242, 'Mexican Peso': 14560, 'Bitcoin':

,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Is Laundering
1138,429848.62,US Dollar,366832.81,Euro,0
1139,27713.70,US Dollar,23650.87,Euro,0
1245,7421.98,US Dollar,6333.92,Euro,0
1246,32261.91,US Dollar,27532.31,Euro,0
1326,9337.12,US Dollar,7968.30,Euro,0



Duplicate rows (full table): 0

Currencies in the dataset:
['Australian Dollar', 'Bitcoin', 'Brazil Real', 'Canadian Dollar', 'Euro', 'Mexican Peso', 'Ruble', 'Rupee', 'Saudi Riyal', 'Shekel', 'Swiss Franc', 'UK Pound', 'US Dollar', 'Yen', 'Yuan']
['Australian Dollar', 'Bitcoin', 'Brazil Real', 'Canadian Dollar', 'Euro', 'Mexican Peso', 'Ruble', 'Rupee', 'Saudi Riyal', 'Shekel', 'Swiss Franc', 'UK Pound', 'US Dollar', 'Yen', 'Yuan']


# Data Preprocessing

In [6]:
#added amount column amount in australian dollars
from forex_python.converter import CurrencyRates
from forex_python.bitcoin import BtcConverter
from currency_converter import CurrencyConverter
import numpy as np
c = CurrencyRates()
b = BtcConverter()
cc = CurrencyConverter()
RATES = {   
        "US Dollar": 1.4,              
        "Euro": 1.64,
        "Bitcoin": 90000,
        "Yuan": 0.2,
        "Yen": 0.0067,
        "UK Pound": 2.0,
        "Brazil Real": 0.2,
        "Australian Dollar": 1.0,
        "Rupee": 0.01,
        "Ruble": 0.01,
        "Canadian Dollar": 1.25,
        "Mexican Peso": 0.07,
        "Swiss Franc": 1.5,            
        "Shekel": 0.25,    
        "Saudi Riyal": 0.25
        }

CODES = {
    "US Dollar": "USD",
    "Euro": "EUR",
    "Yuan": "CNY",
    "Yen": "JPY",
    "UK Pound": "GBP",
    "Brazil Real": "BRL",
    "Australian Dollar": "AUD",
    "Rupee": "INR",
    "Ruble": "RUB",
    "Canadian Dollar": "CAD",
    "Mexican Peso": "MXN",
    "Swiss Franc": "CHF",
    "Shekel": "ILS",
    "Saudi Riyal": "SAR"
}

for i in RATES:
    if i == "Bitcoin":
        try:
            RATES[i] = b.get_latest_price('AUD')
        except:
            pass
    else:
        try:
            RATES[i] = cc.convert(CODES[i], 'AUD', 1)
        except Exception:
            try:
                RATES[i] = c.convert(CODES[i], 'AUD', 1)
            except:
                pass
            
print(RATES)
print("--------------------------------")
print("Currency before conversion")
print("--------------------------------")
print(df[["Amount Paid", "Payment Currency", "Amount Received", "Receiving Currency"]].head())

df["Amount Paid"] = df["Amount Paid"] * df["Payment Currency"].map(RATES)
df["Amount Received"] = df["Amount Received"] * df["Receiving Currency"].map(RATES)
print("--------------------------------")
print("Currency After conversion")
print("--------------------------------")
df["amount"] = (df["Amount Paid"] + df["Amount Received"]) / 2
print(df[["Amount Paid", "Payment Currency", "Amount Received", "Receiving Currency", "amount"]].head())
print(df["amount"].isna().sum())

# Edge flags + timestamp (Payment Format one-hot is fit on TRAIN only in the graph cell)
data = df.copy()
data["Timestamp"] = pd.to_datetime(data["Timestamp"], format="%Y/%m/%d %H:%M", errors="coerce")
data["is_self_transfer"] = (data["sender_id"] == data["receiver_id"]).astype(float)
data["is_cross_bank"] = (data["From Bank"] != data["To Bank"]).astype(float)
data["is_cross_currency"] = (
    data["Payment Currency"] != data["Receiving Currency"]
).astype(float)



{'US Dollar': 1.406185, 'Euro': 1.6278, 'Bitcoin': 90000, 'Yuan': 0.208561, 'Yen': 0.008805, 'UK Pound': 1.901969, 'Brazil Real': 0.270035, 'Australian Dollar': 1.0, 'Rupee': 0.014696, 'Ruble': 0.01311, 'Canadian Dollar': 1.013574, 'Mexican Peso': 0.082457, 'Swiss Franc': 1.730597, 'Shekel': 0.469987, 'Saudi Riyal': 0.25}
--------------------------------
Currency before conversion
--------------------------------
   Amount Paid Payment Currency  Amount Received Receiving Currency
0        27.56        US Dollar            27.56          US Dollar
1      5105.92        US Dollar          5105.92          US Dollar
2   1135598.87        US Dollar       1135598.87          US Dollar
3    614545.68        US Dollar        614545.68          US Dollar
4    105352.58        US Dollar        105352.58          US Dollar
--------------------------------
Currency After conversion
--------------------------------
    Amount Paid Payment Currency  Amount Received Receiving Currency  \
0  3.875446

# Converting to Graph

In [7]:
import numpy as np
import pandas as pd
import torch
import math
from datetime import datetime
from sklearn.preprocessing import OneHotEncoder, RobustScaler

# HI-Medium primary window (Kaggle). Rows after this are ~100% laundering — do not use them in the split.
PRIMARY_START = datetime(2022, 9, 1)
PRIMARY_END = datetime(2022, 9, 16, 23, 59)

n_before = len(data)
post = data.loc[data["Timestamp"] > PRIMARY_END]
print(
    f"Post-window rows dropped: {len(post):,} | "
    f"laundering rate={post['Is Laundering'].mean() * 100 if len(post) else float('nan'):.2f}%"
)
data = data.loc[
    (data["Timestamp"] >= PRIMARY_START) & (data["Timestamp"] <= PRIMARY_END)
].copy()
print(f"Primary-window rows: {len(data):,} (was {n_before:,})")
if data.empty:
    raise ValueError("No rows left after applying the 1–16 Sep 2022 primary window.")

# ---- chrono split (on primary window only)
data = data.sort_values("Timestamp").reset_index(drop=True)
t0 = data["Timestamp"].min()
data["hours_from_start"] = (data["Timestamp"] - t0).dt.total_seconds() / 3600.0
y = data["Is Laundering"].astype(int).to_numpy()

n = len(data)
n_train = int(round(n * 0.70))
n_val = int(round(n * 0.20))

idx_train = np.arange(0, n_train)
idx_val = np.arange(n_train, n_train + n_val)
idx_test = np.arange(n_train + n_val, n)

t_train_end = data.loc[n_train - 1, "Timestamp"]
t_val_end = data.loc[n_train + n_val - 1, "Timestamp"]
print("train ends:", t_train_end)
print("val ends:  ", t_val_end)
print("test ends: ", data["Timestamp"].max())

for name, idx in [("train", idx_train), ("val", idx_val), ("test", idx_test)]:
    rate = data.loc[idx, "Is Laundering"].mean() * 100
    print(f"{name}: fraud rate = {rate:.2f}%")

# --- Edge features (same recipe as EDA-IBM): train-only scaler / one-hot ---
CONT_COLS = ["amount", "hours_from_start"]
CAT_COLS = ["Payment Format"]
BIN_COLS = ["is_self_transfer", "is_cross_bank", "is_cross_currency"]

X_cont = data[CONT_COLS].astype(float).to_numpy()
X_cont_scaled = X_cont.copy()
scaler = RobustScaler()
X_cont_scaled[idx_train] = scaler.fit_transform(X_cont[idx_train])
X_cont_scaled[idx_val] = scaler.transform(X_cont[idx_val])
X_cont_scaled[idx_test] = scaler.transform(X_cont[idx_test])
X_cont_scaled = np.clip(X_cont_scaled, -10, 10)

try:
    cat_encoder = OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)
except TypeError:
    cat_encoder = OneHotEncoder(drop="first", handle_unknown="ignore", sparse=False)
X_cat_raw = data[CAT_COLS].astype(str).to_numpy()
cat_encoder.fit(X_cat_raw[idx_train])
X_cat = cat_encoder.transform(X_cat_raw).astype(np.float32)
cat_names = list(cat_encoder.get_feature_names_out(CAT_COLS))

X_bin = data[BIN_COLS].astype(float).to_numpy()
edge_features = np.hstack([X_cont_scaled.astype(np.float32), X_cat, X_bin])
feature_names = CONT_COLS + cat_names + BIN_COLS
print("edge feature_names:", feature_names)




# --- Node mapping + edge_index from DATAFRAME ORDER (critical)
all_nodes = pd.concat([data["sender_id"], data["receiver_id"]]).unique()
node_mapping = {name: i for i, name in enumerate(all_nodes)}
src = data["sender_id"].map(node_mapping).to_numpy()
dst = data["receiver_id"].map(node_mapping).to_numpy()
edge_index = torch.tensor(np.array([src, dst]), dtype=torch.long)
edge_attr = torch.tensor(edge_features, dtype=torch.float)
edge_labels = torch.tensor(y, dtype=torch.float)
n = len(data)
#----Create Masks---
train_mask = torch.zeros(n, dtype=torch.bool)
val_mask = torch.zeros(n, dtype=torch.bool)
test_mask = torch.zeros(n, dtype=torch.bool)
train_mask[idx_train] = True
val_mask[idx_val] = True
test_mask[idx_test] = True


n_nodes = len(node_mapping)
src_ids = data["sender_id"].map(node_mapping).to_numpy()
dst_ids = data["receiver_id"].map(node_mapping).to_numpy()
amt = data["amount"].to_numpy()

src_tr, dst_tr = src_ids[idx_train], dst_ids[idx_train]
amt_tr = amt[idx_train]

out_cnt = np.zeros(n_nodes)
in_cnt = np.zeros(n_nodes)
out_sum = np.zeros(n_nodes)
in_sum = np.zeros(n_nodes)
np.add.at(out_cnt, src_tr, 1)
np.add.at(in_cnt, dst_tr, 1)
np.add.at(out_sum, src_tr, amt_tr)
np.add.at(in_sum, dst_tr, amt_tr)

node_features = np.stack(
    [
        np.log1p(out_cnt),
        np.log1p(in_cnt),
        np.log1p(out_sum),
        np.log1p(in_sum),
    ],
    axis=1,
)

train_nodes = np.unique(np.concatenate([src_tr, dst_tr]))
node_scaler = RobustScaler()
node_scaler.fit(node_features[train_nodes])
node_features = np.clip(node_scaler.transform(node_features), -10, 10)
node_features = torch.tensor(node_features, dtype=torch.float)









pos_weight_train = (y[idx_train] == 0).sum() / max((y[idx_train] == 1).sum(), 1)
print(f"nodes={len(node_mapping):,} | edges={n:,} | edge_attr_dim={edge_attr.shape[1]}")
print(f"pos_weight (train): {pos_weight_train:.2f}")
for name, m in [("train", train_mask), ("val", val_mask), ("test", test_mask)]:
    yy = y[m.numpy()]
    print(f"{name:5s} | n={len(yy):6d} | fraud={yy.sum():5d} | rate={yy.mean()*100:.3f}%")
# Sanity: all aligned
assert edge_index.shape[1] == edge_attr.shape[0] == edge_labels.shape[0] == n

torch.save(
    {
        "edge_index": edge_index,
        "edge_attr": edge_attr,
        "edge_labels": edge_labels,
        "train_mask": train_mask,
        "val_mask": val_mask,
        "test_mask": test_mask,
        "node_features": node_features,
        "num_nodes": len(node_mapping),
        "node_mapping": node_mapping,
        "pos_weight_train": float(pos_weight_train),
    },
    "ibm_graph.pt",
)
print("Saved graph tensors to ibm_graph.pt")

Post-window rows dropped: 4,552 | laundering rate=89.72%
Primary-window rows: 700,048 (was 704,600)
train ends: 2022-09-12 21:41:00
val ends:   2022-09-15 22:03:00
test ends:  2022-09-16 23:59:00
train: fraud rate = 4.45%
val: fraud rate = 4.87%
test: fraud rate = 3.60%
edge feature_names: ['amount', 'hours_from_start', 'Payment Format_Bitcoin', 'Payment Format_Cash', 'Payment Format_Cheque', 'Payment Format_Credit Card', 'Payment Format_Reinvestment', 'Payment Format_Wire', 'is_self_transfer', 'is_cross_bank', 'is_cross_currency']
nodes=203,096 | edges=700,048 | edge_attr_dim=11
pos_weight (train): 21.47
train | n=490034 | fraud=21808 | rate=4.450%
val   | n=140010 | fraud= 6815 | rate=4.868%
test  | n= 70004 | fraud= 2523 | rate=3.604%
Saved graph tensors to ibm_graph.pt


# Train model

In [ ]:
import pennylane as qml
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import TransformerConv
from sklearn.metrics import average_precision_score
import optuna
import torch.optim as optim
import math



n_qubits = 4
dev = qml.device("default.qubit", wires=n_qubits)
print("Set up device")
@qml.qnode(dev, interface="torch")
def quantum_circuit(inputs, weights):
    for i in range(n_qubits):
        qml.RY(inputs[..., i], wires=i)
    qml.templates.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

class QuantumLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.qlayer = qml.qnn.TorchLayer(
            quantum_circuit, {"weights": (3, n_qubits, 3)}
        )

    def forward(self, x):
        # x: [num_edges, n_qubits] — apply circuit per edge
        return self.qlayer(x)
    



class QGNN(nn.Module):
    def __init__(self, num_nodes, node_attr_dim, edge_attr_dim, hidden_dim):
        super().__init__()
        self.node_enc = nn.Linear(node_attr_dim, hidden_dim)
        self.conv1 = TransformerConv(hidden_dim, hidden_dim, heads=1, edge_dim=edge_attr_dim)
        self.conv2 = TransformerConv(hidden_dim, hidden_dim, heads=1, edge_dim=edge_attr_dim)
        self.to_qubits = nn.Linear(hidden_dim * 2 + edge_attr_dim, n_qubits)
        self.q_layer = QuantumLayer()
        self.classifier = nn.Linear(n_qubits, 1)

    def forward(self, x, mp_edge_index, mp_edge_attr, edge_index, edge_attr, edge_batch):
        x = F.relu(self.node_enc(x))
        x = F.dropout(x, p=0.3, training=self.training)
        x = F.relu(self.conv1(x, mp_edge_index, mp_edge_attr))
        x = F.dropout(x, p=0.3, training=self.training)
        x = self.conv2(x, mp_edge_index, mp_edge_attr)

        src, dst = edge_index[0, edge_batch], edge_index[1, edge_batch]
        ea = edge_attr[edge_batch]
        angles = math.pi * torch.tanh(
            self.to_qubits(torch.cat([x[src], x[dst], ea], dim=-1))
        )
        return self.classifier(self.q_layer(angles)).squeeze(-1)
    
def predict_edges(net, idx):
    net.eval()
    logits_all, trues = [], []
    with torch.no_grad():
        for start in range(0, idx.size(0), BATCH_SIZE):
            batch = idx[start:start + BATCH_SIZE]
            logits = net(node_features, train_edge_index, train_edge_attr, edge_index, edge_attr, edge_batch=batch)
            logits_all.append(logits.cpu())
            trues.append(edge_labels[batch].cpu())
    logits_all = torch.cat(logits_all)
    trues = torch.cat(trues)
    preds = torch.sigmoid(logits_all).numpy()
    return logits_all, preds, trues.numpy()
    
print("Model Setup")
device = torch.device("cpu")
edge_index = edge_index.to(device)
edge_attr = edge_attr.to(device)
edge_labels = edge_labels.to(device)
train_mask = train_mask.to(device)
val_mask = val_mask.to(device)
test_mask = test_mask.to(device)
num_nodes = len(node_mapping)
BATCH_SIZE = 2048  # drop to 512 if you OOM
train_idx = train_mask.nonzero(as_tuple=False).view(-1)
val_idx = val_mask.nonzero(as_tuple=False).view(-1)
test_idx = test_mask.nonzero(as_tuple=False).view(-1)
node_features = node_features.to(device)
train_edge_index = edge_index[:, train_mask]
train_edge_attr = edge_attr[train_mask]


def objective(trial):
    print("Trial started")
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True)
    # keep dims smaller than classical GNN — quantum layer is expensive
    hidden_dim = trial.suggest_categorical("hidden_dim", [8, 16, 32])
    pos_weight_factor = trial.suggest_float("pos_weight_factor", 0.8, 1.2)
    tuning_model = QGNN(
        num_nodes=num_nodes,
        node_attr_dim=node_features.shape[1],
        edge_attr_dim=train_edge_attr.shape[1],
        hidden_dim=hidden_dim,
    ).to(device)
    optimizer = optim.Adam(tuning_model.parameters(), lr=lr, weight_decay=weight_decay)
    adjusted_pos_weight = torch.tensor(
        [float(pos_weight_train) * pos_weight_factor], device=device
    )
    criterion = nn.BCEWithLogitsLoss(pos_weight=adjusted_pos_weight)
    best_val_auprc = 0.0
    for epoch in range(1, 21):  # fewer epochs than classical — quantum is slow
        print("Epoch started")
        tuning_model.train()
        optimizer.zero_grad()
        perm = train_idx[torch.randperm(train_idx.size(0), device=device)]
        total_loss = 0.0
        n_batches = 0

        for start in range(0, perm.size(0), BATCH_SIZE):
            batch = perm[start:start + BATCH_SIZE]

            optimizer.zero_grad()
            logits = tuning_model(node_features, train_edge_index, train_edge_attr, edge_index, edge_attr, edge_batch=batch)
            loss = criterion(logits, edge_labels[batch])
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            n_batches += 1
        
        
        tuning_model.eval()
        with torch.no_grad():
            val_logits, val_preds, val_true = predict_edges(tuning_model, val_idx)
            val_loss = criterion(val_logits, torch.tensor(val_true)).item()
            val_auprc = average_precision_score(val_true, val_preds)   
        best_val_auprc = max(best_val_auprc, val_auprc)
        trial.report(val_auprc, epoch)
        if trial.should_prune():         
            raise optuna.exceptions.TrialPruned()
    return best_val_auprc
print("Tuning started")
optuna.logging.set_verbosity(optuna.logging.INFO)
study = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=5),
)
study.optimize(objective, n_trials=10)  # start small; raise later
print(f"\nBest Validation AUPRC Achieved: {study.best_value:.4f}")
print("Optimal Hyperparameters:", study.best_params)
print("\n--- Starting Phase 5b: Training Final QGNN ---")
best_params = study.best_params
model = QGNN(
    num_nodes=num_nodes,
    node_attr_dim=node_features.shape[1],
    edge_attr_dim=train_edge_attr.shape[1],
    hidden_dim=best_params["hidden_dim"],
).to(device)
optimizer = optim.Adam(
    model.parameters(),
    lr=best_params["lr"],
    weight_decay=best_params["weight_decay"],
)
pos_weight_tensor = torch.tensor(
    [float(pos_weight_train) * best_params["pos_weight_factor"]], device=device
)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
epochs = 50
best_val_auprc = 0.0
best_state = None

for epoch in range(1, epochs + 1):
    model.train()
    optimizer.zero_grad()
    perm = train_idx[torch.randperm(train_idx.size(0), device=device)]
    total_loss = 0.0
    n_batches = 0

    for start in range(0, perm.size(0), BATCH_SIZE):
        batch = perm[start:start + BATCH_SIZE]

        optimizer.zero_grad()
        logits = model(node_features, train_edge_index, train_edge_attr, edge_index, edge_attr, edge_batch=batch)
        loss = criterion(logits, edge_labels[batch])
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1
        
    if epoch % 10 == 0:
        model.eval()
        with torch.no_grad():
            val_logits, val_preds, val_true = predict_edges(model, val_idx)
            val_loss = criterion(val_logits, torch.tensor(val_true)).item()
            val_auprc = average_precision_score(val_true, val_preds)
        print(
            f"Epoch {epoch:03d}/{epochs} | Train Loss: {loss.item():.4f} | "
            f"Val Loss: {val_loss:.4f} | Val AUPRC: {val_auprc:.4f}"
        )
        if val_auprc > best_val_auprc:
            best_val_auprc = val_auprc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            print(f"  New best val AUPRC: {best_val_auprc:.4f}")

if best_state is not None:
    model.load_state_dict(best_state)
    print(f"Loaded best val weights (AUPRC={best_val_auprc:.4f})")

# show test results
model.eval()
with torch.no_grad():
    test_logits, test_preds, test_true = predict_edges(model, test_idx)
    test_auprc = average_precision_score(test_true, test_preds)
    print(f"Test AUPRC: {test_auprc:.4f}")

checkpoint = {
    "model_state_dict": best_state if best_state is not None else model.state_dict(),
    "num_nodes": num_nodes,
    "node_attr_dim": node_features.shape[1],
    "edge_attr_dim": train_edge_attr.shape[1],
    "hidden_dim": best_params["hidden_dim"],
    "n_qubits": n_qubits,
    "best_params": best_params,
    "pos_weight_train": float(pos_weight_train),
    "best_val_auprc": float(best_val_auprc),
    "test_auprc": float(test_auprc),
    "batch_size": BATCH_SIZE,
}
torch.save(checkpoint, "qgnn_checkpoint.pt")
print("Saved model to qgnn_checkpoint.pt")

[I 2026-08-14 09:18:08,805] A new study created in memory with name: no-name-69ddfdab-c958-490c-89aa-beee335e5304


Set up device
Model Setup
Tuning started
Trial started
Epoch started


: 